In [ ]:
import json, os, subprocess
from pathlib import Path
work=Path("/kaggle/working/training");work.mkdir(exist_ok=True)
sources={'bluff_policy.py': '"""Compact learned Bluff action scorer using own cards and public evidence."""\nfrom collections import Counter, defaultdict\nimport json\nfrom pathlib import Path\n\nRANKS = ("A", "2", "3", "4", "5", "6", "7", "8", "9", "10", "J", "Q", "K")\nPLACE_FEATURES = ["batch_size", "truthful", "bluff", "bluff_pile", "bluff_call_rate",\n                  "bluff_finish", "remaining_groups", "remaining_largest_group", "claimed_rank_held"]\nCHALLENGE_FEATURES = ["bias", "revealed_bluff_rate", "rank_held", "claim_count", "pile_size",\n                      "opponent_finishing", "own_small_hand", "opponent_small_hand", "samples"]\nINITIAL_WEIGHTS = [4., 5., -2., -4., -8., -8., -1., .2, .5,\n                   -3.5, 7., .7, -.3, -.5, 1., -1., .5, 0.]\n\n\nclass Policy:\n    def __init__(self, weights):\n        if len(weights) != len(INITIAL_WEIGHTS):\n            raise ValueError("wrong feature count")\n        self.weights = [float(v) for v in weights]\n        self.processed = 0\n        self.claims = {}\n        self.stats = defaultdict(lambda: [0, 0])\n        self.our_outcomes = [0, 0]\n\n    def update(self, observation):\n        history = observation.get("public_history", [])\n        if len(history) < self.processed:\n            self.processed, self.claims, self.stats, self.our_outcomes = 0, {}, defaultdict(lambda: [0, 0]), [0, 0]\n        for event in history[self.processed:]:\n            kind = event.get("type")\n            identity = event.get("claim_id")\n            if kind == "claim":\n                self.claims[identity] = (event.get("seat"), event.get("count", 1))\n            elif kind in ("claim_accepted", "challenge_resolved"):\n                placer, count = self.claims.pop(identity, (event.get("placer", event.get("seat")), event.get("count", 1)))\n                if placer == observation["seat"]:\n                    self.our_outcomes[0] += 1\n                    self.our_outcomes[1] += int(kind == "challenge_resolved")\n                if kind == "challenge_resolved" and isinstance(event.get("truthful"), bool):\n                    stats = self.stats[(placer, count)]\n                    stats[0] += 1\n                    stats[1] += int(not event["truthful"])\n        self.processed = len(history)\n\n    def candidates(self, observation):\n        hand = observation["hand"]\n        held = Counter(c["rank"] for c in hand)\n        pile = min(observation.get("pile_size", 0), 40) / 20.\n        if observation["phase"] == "CHALLENGE":\n            claim = observation["pending_claim"]\n            count, rank, placer = claim["count"], claim["claimed_rank"], claim["seat"]\n            copies = 4 * observation.get("game_config", {}).get("decks", 1)\n            if count > copies - held[rank]:\n                return [({"type": "challenge"}, [0.] * len(self.weights))]\n            samples, bluffs = self.stats[(placer, count)]\n            posterior = (bluffs + .6) / (samples + 5.)\n            sizes = observation["hand_sizes"]\n            vector = [1., posterior, held[rank] / copies, count / 4., pile,\n                      float(claim.get("provisional_finish", False)), float(len(hand) <= 3),\n                      1. / max(1, sizes[placer]), min(samples, 20) / 20.]\n            return [({"type": "accept"}, [0.] * len(self.weights)),\n                    ({"type": "challenge"}, [0.] * len(PLACE_FEATURES) + vector)]\n        groups = defaultdict(list)\n        for card in hand:\n            groups[card["rank"]].append(card)\n        actions = []\n        # Safe truthful groups are always available. Constructed bluff candidates\n        # use only currently owned references; the count is the actual batch size.\n        for rank, cards in groups.items():\n            actions.append({"type": "place", "card_refs": [c["card_ref"] for c in cards[:4]], "claimed_rank": rank})\n        ordered = sorted(hand, key=lambda c: (held[c["rank"]], RANKS.index(c["rank"])))\n        for count in (2, 3):\n            if len(ordered) >= count:\n                for rank in groups:\n                    actions.append({"type": "place", "card_refs": [c["card_ref"] for c in ordered[:count]], "claimed_rank": rank})\n        call_rate = (self.our_outcomes[1] + 1.) / (self.our_outcomes[0] + 4.)\n        by_ref = {c["card_ref"]: c for c in hand}\n        result = []\n        for action in actions:\n            refs = set(action["card_refs"])\n            rank = action["claimed_rank"]\n            truthful = all(by_ref[ref]["rank"] == rank for ref in refs)\n            remaining = Counter(c["rank"] for c in hand if c["card_ref"] not in refs)\n            vector = [len(refs), float(truthful), float(not truthful), float(not truthful) * pile,\n                      float(not truthful) * call_rate, float(not truthful and len(refs) == len(hand)),\n                      len(remaining) / 13., max(remaining.values(), default=0) / 4., held[rank] / 4.]\n            result.append((action, vector + [0.] * len(CHALLENGE_FEATURES)))\n        return result\n\n    def act(self, observation):\n        self.update(observation)\n        return max(self.candidates(observation), key=lambda candidate:\n                   sum(w * value for w, value in zip(self.weights, candidate[1])))[0]\n\n    def save(self, path, metadata):\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n        Path(path).write_text(json.dumps({"format": "triad-bluff-linear-v1", "features": PLACE_FEATURES + CHALLENGE_FEATURES,\n            "weights": self.weights, "training": metadata}, indent=2) + "\\n")\n', 'bluff_policy_v2.py': '"""Learned Bluff scorer with conservative rank bounds and public caller evidence.\n\nInference reads only own hand/private history and public events. Unknown discarded\ncards decrement every rank bound, ensuring no hidden-hand identity is assumed.\n"""\nfrom collections import Counter, defaultdict\nimport json\nimport math\nfrom pathlib import Path\n\nfrom bluff_policy import Policy as V1, PLACE_FEATURES, CHALLENGE_FEATURES, INITIAL_WEIGHTS as V1_INITIAL, RANKS\n\nEXTRA_PLACE = [\'bluff_any_call_probability\', \'bluff_impossible_for_caller\',\n               \'truthful_turns_removed\', \'bluff_known_pile_fraction\',\n               \'truthful_finish\', \'bluff_no_impossibility_call_possible\']\nEXTRA_CHALLENGE = [\'placer_global_revealed_bluff_rate\', \'known_outside_rank_fraction\',\n                   \'known_pile_fraction\', \'claim_fraction_of_original_hand\',\n                   \'finishing_large_batch\', \'known_placer_rank_before_claim\']\nFEATURE_NAMES = PLACE_FEATURES + CHALLENGE_FEATURES + EXTRA_PLACE + EXTRA_CHALLENGE\nINITIAL_WEIGHTS = [.5, 1., -.5, -2., -8., -3., -1., .2, .5] + V1_INITIAL[len(PLACE_FEATURES):] + [-6., -12., 3., -1., 4., 3., 1., 1., -.5, .5, .5, -.5]\n\n\ndef identity(card):\n    return card.get(\'deck\', 0), card[\'rank\'], card[\'suit\']\n\n\nclass Policy(V1):\n    def __init__(self, weights):\n        if len(weights) != len(FEATURE_NAMES) or not all(math.isfinite(float(w)) for w in weights):\n            raise ValueError(\'invalid model weights\')\n        super().__init__(weights[:len(V1_INITIAL)])\n        self.weights = list(map(float, weights))\n        self._reset_evidence()\n\n    def _reset_evidence(self):\n        self.folded_history = []\n        self.known = defaultdict(Counter)\n        self.pile_known = {}\n        self.calls = defaultdict(lambda: [0, 0])\n        self.global_bluffs = defaultdict(lambda: [0, 0])\n        self.pending_meta = {}\n        self.active = []\n        self.before_claim = {}\n\n    def update(self, observation):\n        history = observation.get(\'public_history\', [])\n        if len(history) < len(self.folded_history) or history[:len(self.folded_history)] != self.folded_history:\n            self.processed = 0\n            self.claims = {}\n            self.stats = defaultdict(lambda: [0, 0])\n            self.our_outcomes = [0, 0]\n            self._reset_evidence()\n        super().update(observation)\n        own_places = {event[\'claim_id\']: event[\'cards\'] for event in observation.get(\'private_history\', []) if event.get(\'type\') == \'own_placement\'}\n        for event in history[len(self.folded_history):]:\n            kind, cid = event.get(\'type\'), event.get(\'claim_id\')\n            if kind == \'game_start\':\n                self.active = list(event[\'active_seats\'])\n            elif kind == \'claim\':\n                placer, count = event[\'seat\'], event[\'count\']\n                self.before_claim[cid] = dict(self.known[placer])\n                # The rank in an unverified claim is not evidence of actual cards.\n                for rank in list(self.known[placer]):\n                    self.known[placer][rank] = max(0, self.known[placer][rank] - count)\n                priority = sorted((s for s in self.active if s != placer), key=lambda s: (s - placer) % observation[\'game_config\'][\'players\'])\n                self.pending_meta[cid] = (placer, count, priority, bool(event.get(\'provisional_finish\')))\n                for card in own_places.get(cid, []):\n                    self.pile_known[identity(card)] = card\n            elif kind in (\'claim_accepted\', \'challenge_resolved\'):\n                placer, count, priority, finish = self.pending_meta.pop(cid)\n                selected = event.get(\'challenger\') if kind == \'challenge_resolved\' else None\n                # The first selected challenger is revealed. Earlier seats must\n                # have accepted; later choices remain unknown and are not counted.\n                for seat in priority:\n                    record = self.calls[(seat, count, finish)]\n                    record[0] += 1\n                    record[1] += int(seat == selected)\n                    if seat == selected:\n                        break\n                if kind == \'challenge_resolved\':\n                    record = self.global_bluffs[placer]\n                    record[0] += 1\n                    record[1] += int(not event[\'truthful\'])\n                    for card in event[\'revealed_batch\']:\n                        self.pile_known[identity(card)] = card\n            elif kind == \'pickup\':\n                recipient = event[\'recipient\']\n                self.known[recipient].update(card[\'rank\'] for card in self.pile_known.values())\n                self.pile_known = {}\n            elif kind == \'finish\':\n                self.active = [s for s in self.active if s != event[\'seat\']]\n        self.folded_history = list(history)\n\n    def caller_probability(self, observation, count, finish):\n        escape = 1.\n        for seat in observation[\'active_seats\']:\n            if seat == observation[\'seat\']:\n                continue\n            samples, calls = self.calls.get((seat, count, finish), (0, 0))\n            # Beta prior avoids certainties from small or selectively observed samples.\n            escape *= 1. - (calls + 1.) / (samples + 5.)\n        return 1. - escape\n\n    def candidates(self, observation):\n        hand = observation[\'hand\']\n        held = Counter(c[\'rank\'] for c in hand)\n        copies = 4 * observation.get(\'game_config\', {}).get(\'decks\', 1)\n        if observation[\'phase\'] == \'CHALLENGE\':\n            claim = observation[\'pending_claim\']\n            placer, rank, count = claim[\'seat\'], claim[\'claimed_rank\'], claim[\'count\']\n            outside = held[rank] + sum(counts[rank] for seat, counts in self.known.items() if seat not in (observation[\'seat\'], placer))\n            if count > copies - outside:\n                return [({\'type\': \'challenge\'}, [0.] * len(self.weights))]\n            base = super().candidates(observation)\n            samples, bluffs = self.global_bluffs.get(placer, (0, 0))\n            extra = [(bluffs + .6) / (samples + 5.), outside / copies,\n                     len(self.pile_known) / max(1, observation[\'pile_size\']),\n                     count / max(1, observation[\'hand_sizes\'][placer] + count),\n                     float(claim.get(\'provisional_finish\', False)) * count / 4.,\n                     self.before_claim.get(claim[\'claim_id\'], {}).get(rank, 0) / copies]\n            return [(a, v + [0.] * len(EXTRA_PLACE) + (extra if a[\'type\'] == \'challenge\' else [0.] * len(EXTRA_CHALLENGE))) for a, v in base]\n        groups = defaultdict(list)\n        for card in hand:\n            groups[card[\'rank\']].append(card)\n        actions = [action for action, _ in super().candidates(observation)]\n        # Full truthful batches plus partial truthful batches and mixed shedding\n        # batches of up to four cards, including a possible finishing bluff.\n        for rank, cards in groups.items():\n            for count in range(1, min(4, len(cards)) + 1):\n                actions.append({\'type\': \'place\', \'card_refs\': [c[\'card_ref\'] for c in cards[:count]], \'claimed_rank\': rank})\n        ordered = sorted(hand, key=lambda c: (held[c[\'rank\']], RANKS.index(c[\'rank\'])))\n        for count in range(2, min(4, len(hand)) + 1):\n            for rank in RANKS:\n                actions.append({\'type\': \'place\', \'card_refs\': [c[\'card_ref\'] for c in ordered[:count]], \'claimed_rank\': rank})\n        by_ref, seen, result = {c[\'card_ref\']: c for c in hand}, set(), []\n        pile = min(observation.get(\'pile_size\', 0), 40) / 20.\n        call_rate = (self.our_outcomes[1] + 1.) / (self.our_outcomes[0] + 4.)\n        for action in actions:\n            refs, rank = set(action[\'card_refs\']), action[\'claimed_rank\']\n            key = tuple(sorted(refs)), rank\n            if key in seen:\n                continue\n            seen.add(key)\n            truthful = all(by_ref[r][\'rank\'] == rank for r in refs)\n            bluff = float(not truthful)\n            remaining = Counter(c[\'rank\'] for c in hand if c[\'card_ref\'] not in refs)\n            finish = not remaining\n            vector = [len(refs), float(truthful), bluff, bluff * pile, bluff * call_rate,\n                      float(bluff and finish), len(remaining) / 13., max(remaining.values(), default=0) / 4., held[rank] / 4.]\n            impossible = any(self.known.get(s, {}).get(rank, 0) + len(refs) > copies for s in observation[\'active_seats\'] if s != observation[\'seat\'])\n            extra = [bluff * self.caller_probability(observation, len(refs), finish),\n                     bluff * impossible, sum((n + 3) // 4 for n in held.values()) - sum((n + 3) // 4 for n in remaining.values()),\n                     bluff * len(self.pile_known) / max(1, observation[\'pile_size\']),\n                     float(truthful and finish), bluff * float(held[rank] >= len(refs))]\n            result.append((action, vector + [0.] * len(CHALLENGE_FEATURES) + extra + [0.] * len(EXTRA_CHALLENGE)))\n        return result\n\n    def save(self, path, metadata):\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n        Path(path).write_text(json.dumps({\'format\': \'triad-bluff-linear-v2\', \'features\': FEATURE_NAMES,\n                                        \'weights\': self.weights, \'training\': metadata}, indent=2) + \'\\n\')\n', 'bluff_policy_v3.py': '"""Bluff v2 model with additional, sound public/own pile rank deductions.\n\nOlder unchallenged batches are never assumed truthful. These deductions use\nonly own placement cards and the last publicly revealed batch before pickup.\n"""\nfrom collections import Counter\nfrom bluff_policy_v2 import Policy as V2, FEATURE_NAMES, INITIAL_WEIGHTS\n\n\nclass Policy(V2):\n    def candidates(self, observation):\n        if observation[\'phase\'] == \'CHALLENGE\':\n            claim = observation[\'pending_claim\']\n            rank, count, placer = claim[\'claimed_rank\'], claim[\'count\'], claim[\'seat\']\n            # Known old pile cards, current own cards and lower bounds on other\n            # players\' hands are disjoint, so their sum is a conservative count.\n            pile_count = Counter(card[\'rank\'] for card in self.pile_known.values())[rank]\n            held_count = sum(card[\'rank\'] == rank for card in observation[\'hand\'])\n            other_count = sum(counts[rank] for seat, counts in self.known.items()\n                              if seat not in (observation[\'seat\'], placer))\n            copies = 4 * observation[\'game_config\'][\'decks\']\n            if count > copies - held_count - other_count - pile_count:\n                return [({\'type\': \'challenge\'}, [0.] * len(self.weights))]\n        return super().candidates(observation)\n', 'train_bluff_league.py': '"""Train against diverse Bluff policies with paired seat blocks and held-out tests."""\nimport argparse\nimport concurrent.futures\nimport json\nimport multiprocessing\nimport os\nfrom pathlib import Path\nimport random\nimport sys\nimport time\nimport numpy as np\n\nROOT = Path(__file__).resolve().parent\nSDK = Path(os.environ.get(\'BLUFF_SDK\', ROOT.parent / \'bluffSDK\'))\nsys.path.insert(0, str(SDK / \'local_evaluator/src\'))\nfrom triad_bluff import engine\nfrom triad_bluff.baselines import HeuristicAgent, RandomAgent, TruthfulAgent, AggressiveChallengeAgent\nfrom bluff_policy import Policy as V1, INITIAL_WEIGHTS as V1_INITIAL, RANKS\nfrom bluff_policy_v2 import Policy as V2, INITIAL_WEIGHTS, FEATURE_NAMES\nfrom bluff_policy_v3 import Policy as V3\n\n\nclass Starter(TruthfulAgent):\n    def act(self, observation):\n        if observation[\'phase\'] == \'CHALLENGE\':\n            return super().act(observation)\n        from collections import defaultdict\n        groups = defaultdict(list)\n        for card in observation[\'hand\']:\n            groups[card[\'rank\']].append(card)\n        rank, cards = max(groups.items(), key=lambda item: (len(item[1]), item[0]))\n        return {\'type\': \'place\', \'card_refs\': [c[\'card_ref\'] for c in cards[:4]], \'claimed_rank\': rank}\n\n\nclass SDKCompetitive:\n    """The SDK\'s supplied adaptive strategy, used only as a development rival."""\n    def __init__(self):\n        import importlib.util\n        path = SDK / \'agents/bluff-agent/src/strategy.py\'\n        spec = importlib.util.spec_from_file_location(\'sdk_competitive_strategy\', path)\n        module = importlib.util.module_from_spec(spec)\n        spec.loader.exec_module(module)\n        self.select_action = module.select_action\n        self.state = None\n\n    def act(self, observation):\n        action, self.state = self.select_action(observation, self.state)\n        return action\n\n\nclass StochasticBluffer(TruthfulAgent):\n    def __init__(self, seed, bluff_rate, call_rate):\n        self.rng = random.Random(seed)\n        self.bluff_rate, self.call_rate = bluff_rate, call_rate\n\n    def act(self, observation):\n        truthful = super().act(observation)\n        if observation[\'phase\'] == \'CHALLENGE\':\n            if truthful[\'type\'] == \'challenge\':\n                return truthful\n            finishing = observation[\'pending_claim\'].get(\'provisional_finish\', False)\n            probability = min(.95, self.call_rate + .2 * finishing)\n            return {\'type\': \'challenge\' if self.rng.random() < probability else \'accept\'}\n        hand = observation[\'hand\']\n        if len(hand) > 1 and self.rng.random() < self.bluff_rate:\n            from collections import Counter\n            held = Counter(c[\'rank\'] for c in hand)\n            count = min(4, len(hand)) if len(hand) <= 4 else self.rng.choice((2, 3, 4))\n            chosen = sorted(hand, key=lambda c: (held[c[\'rank\']], RANKS.index(c[\'rank\'])))[:count]\n            # Owning copies of the claimed rank makes impossible calls less likely.\n            claimed = max(RANKS, key=lambda rank: held[rank])\n            return {\'type\': \'place\', \'card_refs\': [c[\'card_ref\'] for c in chosen], \'claimed_rank\': claimed}\n        return truthful\n\n\ndef make_policy(spec, seed):\n    kind = spec[\'kind\']\n    if kind == \'v1\':\n        return V1(spec[\'weights\'])\n    if kind == \'v2\':\n        return V2(spec[\'weights\'])\n    if kind == \'v3\':\n        return V3(spec[\'weights\'])\n    if kind == \'bluffer\':\n        return StochasticBluffer(seed, spec[\'bluff_rate\'], spec[\'call_rate\'])\n    if kind == \'starter\':\n        return Starter()\n    if kind == \'competitive\':\n        return SDKCompetitive()\n    if kind == \'truthful\':\n        return TruthfulAgent()\n    if kind == \'random\':\n        return RandomAgent(seed)\n    if kind == \'aggressive\':\n        return AggressiveChallengeAgent(seed)\n    return HeuristicAgent(seed)\n\n\ndef score_game(seed, seat, candidate, rivals, horizon=1000):\n    actors = [None] * 4\n    actors[seat] = make_policy(candidate, seed * 10 + 9)\n    for offset, rival in enumerate(rivals, 1):\n        actors[(seat + offset) % 4] = make_policy(rival, seed * 10 + offset)\n    state = engine.new_game(engine.GameConfig(max_turns=horizon), seed)\n    while state.outcome is None and seat not in state.finished_set:\n        decision = engine.next_decision(state)\n        observation = engine.observe(state, decision.seat)\n        payload = actors[decision.seat].act(observation)\n        action = engine.decode_action(state, decision, payload)\n        if not engine.step(state, decision, action).ok:\n            raise RuntimeError(\'policy produced an illegal action\')\n    if seat in state.finished_order:\n        placement = state.finished_order.index(seat)\n        return state.config.position_points[placement], float(placement == 0)\n    return state.outcome.points[seat], 0.\n\n\ndef evaluate_job(job):\n    candidate, seeds, pool, mode, *options = job\n    horizon = options[0] if options else 1000\n    points, wins = [], []\n    for seed in seeds:\n        rng = random.Random(seed ^ 0x424C554646)\n        if mode == \'league\':\n            rivals = [rng.choice(pool) for _ in range(3)]\n        elif mode == \'bluffer\':\n            rivals = [{\'kind\': \'bluffer\', \'bluff_rate\': .35, \'call_rate\': .1}] * 3\n        else:\n            rivals = [{\'kind\': mode}] * 3\n        outcomes = [score_game(seed, seat, candidate, rivals, horizon) for seat in range(4)]\n        points.append(sum(p for p, _ in outcomes) / 4.)\n        wins.append(sum(w for _, w in outcomes) / 4.)\n    return {\'points\': float(np.mean(points)), \'win_rate\': float(np.mean(wins)),\n            \'games\': 4 * len(seeds), \'block_points\': points, \'block_wins\': wins}\n\n\ndef summarize(result):\n    return {k: v for k, v in result.items() if not k.startswith(\'block_\')}\n\n\ndef paired_interval(candidate, reference, seed=20261014):\n    differences = np.asarray(candidate[\'block_points\']) - np.asarray(reference[\'block_points\'])\n    rng = np.random.default_rng(seed)\n    means = differences[rng.integers(0, len(differences), (5000, len(differences)))].mean(axis=1)\n    return {\'mean_gain\': float(differences.mean()), \'ci95\': [float(v) for v in np.quantile(means, [.025, .975])],\n            \'unit\': \'points per game\', \'sampling_unit\': \'four-seat seed block\'}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--incumbent\', type=Path)\n    parser.add_argument(\'--reference-policy\', type=Path)\n    parser.add_argument(\'--policy-kind\', choices=(\'v2\', \'v3\'), default=\'v2\')\n    parser.add_argument(\'--selection-seed-start\', type=int, default=8_000_000)\n    parser.add_argument(\'--test-seed-start\', type=int, default=9_000_000)\n    parser.add_argument(\'--train-seed-start\', type=int, default=50_000)\n    parser.add_argument(\'--iterations\', type=int, default=30)\n    parser.add_argument(\'--population\', type=int, default=32)\n    parser.add_argument(\'--train-blocks\', type=int, default=8)\n    parser.add_argument(\'--train-horizon\', type=int, default=200)\n    parser.add_argument(\'--validation-blocks\', type=int, default=64)\n    parser.add_argument(\'--test-blocks\', type=int, default=256)\n    parser.add_argument(\'--workers\', type=int, default=min(4, os.cpu_count() or 1))\n    parser.add_argument(\'--seed\', type=int, default=20261016)\n    parser.add_argument(\'--output\', type=Path, default=ROOT / \'bluff-league-output\')\n    args = parser.parse_args()\n    if min(args.iterations, args.population, args.train_blocks, args.validation_blocks, args.test_blocks, args.workers, args.train_horizon) < 1 or args.train_horizon > 1000:\n        parser.error(\'counts must be positive\')\n    args.output.mkdir(parents=True, exist_ok=True)\n    weights = json.loads(args.incumbent.read_text())[\'weights\'] if args.incumbent else V1_INITIAL\n    incumbent = {\'kind\': args.policy_kind if len(weights) == len(FEATURE_NAMES) else \'v1\', \'weights\': weights}\n    mean = np.asarray(weights + INITIAL_WEIGHTS[len(weights):], dtype=float)\n    best, sigma = mean.copy(), np.full(len(mean), 1.5)\n    rng = np.random.default_rng(args.seed)\n    cautious = list(INITIAL_WEIGHTS); cautious[2] -= 8.; cautious[4] -= 8.\n    daring = list(INITIAL_WEIGHTS); daring[2] += 4.; daring[5] += 4.\n    reference = [{\'kind\': k} for k in (\'starter\', \'heuristic\', \'truthful\', \'aggressive\', \'random\', \'competitive\')] + [\n        incumbent, {\'kind\': \'v2\', \'weights\': INITIAL_WEIGHTS}, {\'kind\': \'v2\', \'weights\': cautious},\n        {\'kind\': \'v2\', \'weights\': daring}] + [\n        {\'kind\': \'bluffer\', \'bluff_rate\': b, \'call_rate\': c} for b,c in ((.15,.03),(.35,.1),(.5,.35))]\n    if args.reference_policy:\n        reference_document = json.loads(args.reference_policy.read_text())\n        reference.append({\'kind\': \'v1\' if len(reference_document[\'weights\']) == len(V1_INITIAL) else \'v2\',\n                          \'weights\': reference_document[\'weights\']})\n    archive = list(reference)\n    selection_seeds = list(range(args.selection_seed_start, args.selection_seed_start + args.validation_blocks))\n    history, started = [], time.time()\n    with concurrent.futures.ProcessPoolExecutor(max_workers=args.workers, mp_context=multiprocessing.get_context(\'spawn\')) as pool:\n        baseline = list(pool.map(evaluate_job, [(incumbent, selection_seeds, reference, \'league\')]))[0]\n        initial = list(pool.map(evaluate_job, [({\'kind\': args.policy_kind, \'weights\': best.tolist()}, selection_seeds, reference, \'league\')]))[0]\n        selected_score = initial[\'points\']\n        for generation in range(args.iterations):\n            if generation and generation % 10 == 0:\n                mean, sigma = best.copy(), np.full(len(mean), 1.5)\n            population = rng.normal(mean, sigma, (args.population, len(mean)))\n            probes = max(1, args.population // 5)\n            population[-probes:] = rng.normal(best, 3., (probes, len(mean)))\n            population = np.clip(population, -20., 20.)\n            population[0] = mean\n            if args.population > 1:\n                population[1] = best\n            seeds = list(range(args.train_seed_start + generation * args.train_blocks, args.train_seed_start + (generation + 1) * args.train_blocks))\n            # Fast training probes use a shorter cap; model selection and final\n            # reporting always use the real 1,000-turn cap. Short-cap results are\n            # approximate training rewards, never evidence of event performance.\n            scores = list(pool.map(evaluate_job, [({\'kind\':args.policy_kind,\'weights\':w.tolist()},seeds,archive,\'league\',args.train_horizon) for w in population]))\n            ranked = sorted(range(len(scores)), key=lambda i: (scores[i][\'points\'], scores[i][\'win_rate\']), reverse=True)\n            elite = population[ranked[:max(2, args.population // 5)]]\n            mean = .35 * mean + .65 * elite.mean(axis=0)\n            sigma = np.maximum(.3, .35 * sigma + .65 * elite.std(axis=0))\n            candidate = population[ranked[0]]\n            spec = {\'kind\':args.policy_kind,\'weights\':candidate.tolist()}\n            validation = list(pool.map(evaluate_job, [(spec,selection_seeds,reference,\'league\')]))[0]\n            if validation[\'points\'] > selected_score:\n                best, selected_score = candidate.copy(), validation[\'points\']\n            if generation % 4 == 3:\n                archive.append(spec)\n                archive = reference + archive[len(reference):][-12:]\n            record = {\'generation\':generation+1,\'train\':summarize(scores[ranked[0]]),\'validation\':summarize(validation),\n                      \'paired_validation_gain\':paired_interval(validation,baseline),\'selected_points\':selected_score,\n                      \'archive_size\':len(archive),\'elapsed_seconds\':round(time.time()-started,1)}\n            history.append(record); print(json.dumps(record),flush=True)\n            V2(best).save(args.output/\'bluff-policy-v2.json\',{\'method\':\'archive-league cross-entropy policy optimization\',\n                \'policy_kind\': args.policy_kind, \'selection_seed_start\': args.selection_seed_start,\n                \'test_seed_start\': args.test_seed_start, \'train_seed_start\': args.train_seed_start,\n                \'seed\':args.seed,\'generations\':generation+1,\'population\':args.population,\n                \'train_blocks\':args.train_blocks,\'train_horizon\':args.train_horizon,\'evaluation_horizon\':1000,\'validation_blocks\':args.validation_blocks,\n                \'selected_validation_points\':selected_score,\'incumbent_validation_points\':baseline[\'points\'],\n                \'sdk_commit\':\'96da8042c285af4e4f43655f2ad99592bd49b8e6\'})\n            (args.output/\'training-progress.json\').write_text(json.dumps(history,indent=2)+\'\\n\')\n            (args.output/\'archive.json\').write_text(json.dumps(archive,indent=2)+\'\\n\')\n        seeds = list(range(args.test_seed_start,args.test_seed_start+args.test_blocks))\n        chosen = {\'kind\':args.policy_kind,\'weights\':best.tolist()}\n        modes = (\'league\',\'starter\',\'truthful\',\'bluffer\',\'aggressive\',\'competitive\')\n        results = list(pool.map(evaluate_job,[(spec,seeds,reference,mode) for mode in modes for spec in (chosen,incumbent)]))\n        report = {mode:{\'candidate\':summarize(results[2*i]),\'incumbent\':summarize(results[2*i+1]),\n                        \'paired_gain\':paired_interval(results[2*i],results[2*i+1])} for i,mode in enumerate(modes)}\n        (args.output/\'evaluation.json\').write_text(json.dumps(report,indent=2)+\'\\n\')\n        print(json.dumps({\'held_out_evaluation\':report}),flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n'}
for name,source in sources.items(): (work/name).write_text(source)
(work/"incumbent.json").write_text('{\n  "format": "triad-bluff-linear-v2",\n  "features": [\n    "batch_size",\n    "truthful",\n    "bluff",\n    "bluff_pile",\n    "bluff_call_rate",\n    "bluff_finish",\n    "remaining_groups",\n    "remaining_largest_group",\n    "claimed_rank_held",\n    "bias",\n    "revealed_bluff_rate",\n    "rank_held",\n    "claim_count",\n    "pile_size",\n    "opponent_finishing",\n    "own_small_hand",\n    "opponent_small_hand",\n    "samples",\n    "bluff_any_call_probability",\n    "bluff_impossible_for_caller",\n    "truthful_turns_removed",\n    "bluff_known_pile_fraction",\n    "truthful_finish",\n    "bluff_no_impossibility_call_possible",\n    "placer_global_revealed_bluff_rate",\n    "known_outside_rank_fraction",\n    "known_pile_fraction",\n    "claim_fraction_of_original_hand",\n    "finishing_large_batch",\n    "known_placer_rank_before_claim"\n  ],\n  "weights": [\n    0.4755931924211193,\n    5.820676470779726,\n    -3.445920468550993,\n    0.5025301512894197,\n    -10.204982725387449,\n    -10.63672367105182,\n    1.9122839363624424,\n    6.692718730179745,\n    3.3398384247884447,\n    -3.967913294968817,\n    7.18093879068249,\n    -8.248647532051631,\n    -4.252488827029067,\n    -5.717390427387336,\n    -10.297751467694116,\n    -4.172529040138095,\n    3.643993344291045,\n    4.785033532440487,\n    -8.90940656773177,\n    -9.18268199666696,\n    3.5590525036076692,\n    -1.533025684746697,\n    5.957038223067108,\n    9.714173152584005,\n    3.5582377937169647,\n    -0.47803421898358756,\n    -1.2639464162072949,\n    0.6168380283723138,\n    2.861924827690982,\n    -7.115720155350792\n  ],\n  "training": {\n    "method": "archive-league cross-entropy policy optimization",\n    "seed": 20261016,\n    "generations": 30,\n    "population": 32,\n    "train_blocks": 8,\n    "train_horizon": 200,\n    "evaluation_horizon": 1000,\n    "validation_blocks": 64,\n    "selected_validation_points": 4.0859375,\n    "incumbent_validation_points": 3.62109375,\n    "sdk_commit": "96da8042c285af4e4f43655f2ad99592bd49b8e6"\n  }\n}\n')
(work/"reference.json").write_text('{\n  "format": "triad-bluff-linear-v1",\n  "features": [\n    "batch_size",\n    "truthful",\n    "bluff",\n    "bluff_pile",\n    "bluff_call_rate",\n    "bluff_finish",\n    "remaining_groups",\n    "remaining_largest_group",\n    "claimed_rank_held",\n    "bias",\n    "revealed_bluff_rate",\n    "rank_held",\n    "claim_count",\n    "pile_size",\n    "opponent_finishing",\n    "own_small_hand",\n    "opponent_small_hand",\n    "samples"\n  ],\n  "weights": [\n    4.351868833686492,\n    5.6322712663248415,\n    -2.6260873498626167,\n    0.7063932700439369,\n    -10.84185815552825,\n    -8.300750779890834,\n    -0.7486711333834275,\n    1.2260583122844886,\n    -1.5575700886723332,\n    -4.558178606791033,\n    8.25750430388175,\n    0.43873901259948533,\n    -1.3556124209301408,\n    -1.1333529834957075,\n    0.14766041336362645,\n    -1.5734330345566885,\n    -0.4730921181099686,\n    -0.5811911507935986\n  ],\n  "training": {\n    "method": "cross-entropy policy optimization",\n    "seed": 20261014,\n    "generations": 25,\n    "population": 32,\n    "games_per_candidate": 64,\n    "validation": 3.86328125,\n    "sdk_commit": "96da8042c285af4e4f43655f2ad99592bd49b8e6"\n  }\n}\n')
sdk=Path("/kaggle/working/bluffSDK")
subprocess.run(["git","clone","--quiet","https://github.com/Sir-Nike/bluffSDK",str(sdk)],check=True)
subprocess.run(["git","-C",str(sdk),"checkout","--quiet","96da8042c285af4e4f43655f2ad99592bd49b8e6"],check=True)
os.environ["BLUFF_SDK"]=str(sdk)
subprocess.run(['python', '/kaggle/working/training/train_bluff_league.py', '--incumbent', '/kaggle/working/training/incumbent.json', '--reference-policy', '/kaggle/working/training/reference.json', '--policy-kind', 'v3', '--iterations', '40', '--population', '32', '--train-blocks', '16', '--train-horizon', '200', '--validation-blocks', '128', '--test-blocks', '256', '--workers', '4', '--selection-seed-start', '25000000', '--test-seed-start', '26000000', '--train-seed-start', '24000000', '--output', '/kaggle/working/output'],check=True,timeout=18000)
